# Notebook companion — LLM Ops in produzione

Questo notebook segue la demo live dello stack LLM Ops. Non contiene business logic: ogni cella importa codice da `llm_ops_v1`, così il notebook resta una mappa didattica e non una seconda applicazione.

**Scenario:** un ticket di supporto entra nel sistema, l'agente lo classifica, stima il costo, controlla la cache, genera una risposta e produce segnali osservabili.

> Le celle marcate `[OFFLINE]` funzionano senza chiavi API.  
> Le celle marcate `[LIVE]` richiedono almeno `ANTHROPIC_API_KEY`.

In [ ]:
# [OFFLINE] Setup — eseguire sempre per prima
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

SRC_PATH = REPO_ROOT / "src"
sys.path.insert(0, str(SRC_PATH))

load_dotenv(REPO_ROOT / ".env", override=False)

has_api = bool(os.getenv("ANTHROPIC_API_KEY"))
has_lf = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))
langfuse_status = "non impostata (tracing disabilitato)"
if has_lf:
    langfuse_status = "set — " + os.getenv("LANGFUSE_HOST", "")
print(f"repo root : {REPO_ROOT}")
print(f"src path  : {SRC_PATH}")
print(f"API key   : {'set' if has_api else 'non impostata (modalita offline)'}")
print(f"Langfuse  : {langfuse_status}")

## Scenario: repo e agente

Usiamo un ticket di supporto perché ha input, policy, decisione e rischio operativo, piccolo per la demo.

In [ ]:
# [OFFLINE + LIVE] — usa il modello reale se ANTHROPIC_API_KEY e impostata
import os

from IPython.display import Markdown, display

from llm_ops_v1.agents.base_agent import SupportTriageDependencies, run_triage_with_usage
from llm_ops_v1.observability.langfuse_setup import flush_langfuse

PROMPT = (
    "Il mio ordine e' in ritardo. Prepara una risposta di supporto, "
    "cita l'aggiornamento di tracking e decidi se escalare."
)
DEPS = SupportTriageDependencies(
    ticket_id="ticket-demo-042",
    customer_tier="priority",
    ticket_priority="high",
    policy_snippets=["SLA priority: inviare un aggiornamento entro 4 ore."],
)

result = await run_triage_with_usage(PROMPT, deps=DEPS)
TRIAGE_OUTPUT = result.output
flush_langfuse()

mode = (
    "OFFLINE (stima)"
    if result.estimated
    else f"LIVE  in={result.input_tokens} out={result.output_tokens} tok"
)
print(f"[{mode}]")
print()
display(Markdown(TRIAGE_OUTPUT))

## Costi e routing

Domanda operativa: quanto costa questo ticket, e quando conviene cambiare modello? Mostriamo pricing per provider e il routing cost-aware che sceglie automaticamente in base a complessita e budget.

In [ ]:
# [OFFLINE] Pricing per provider
from llm_ops_v1.economics.commercial_models import get_pricing, list_commercial_models
from llm_ops_v1.economics.cost_calculator import estimate_token_cost
from llm_ops_v1.economics.local_models import LOCAL_MODELS, estimate_local_infra_cost

print("Modelli disponibili:", list_commercial_models())
print()
for mid in [
    "anthropic:claude-haiku-4-5",
    "anthropic:claude-sonnet-4-6",
    "gemini:gemini-3.5-flash",
    "openai:gpt-5.5",
    "openrouter:deepseek-v4-flash",
]:
    c = estimate_token_cost(get_pricing(mid), 1200, 350, cached_input_tokens=800)
    print(f"  {mid:45s}  ${c.total_cost_usd:.6f}")

print()
LOCAL_MODEL = "ollama:qwen3.6:27b"
latency = 1.4
local = estimate_local_infra_cost("ollama:qwen3.6:27b", latency_seconds=latency)
profile = LOCAL_MODELS[LOCAL_MODEL]
print(f"  Infra locale ({LOCAL_MODEL}, {latency}s): ${local:.6f}")
print()
print("Come viene calcolato il costo locale:")
hourly = profile.estimated_hourly_infra_usd
print(f"  hourly_rate = ${hourly:.2f}/h")
print("  include elettricita + quota hardware su 3 anni")
print(f"  costo       = {latency}s / 3600s × ${hourly:.2f} = ${local:.6f}")
print()
print("Il costo orario ($0.40/h per qwen3.6:27b) stima il costo marginale di tenere")
print("l'hardware acceso e ammortizzato: GPU/CPU electricity + quota hardware su 3 anni.")

In [ ]:
# [OFFLINE] Routing cost-aware — decide_form sceglie modello per complessita + budget
from llm_ops_v1.agents.router import decide_form, route_trace

tickets = [
    "Dove e' il mio pacco?",
    "Enterprise: billing refund e app outage bloccano il lancio — urgente.",
    "Help.",
]
for t in tickets:
    form = decide_form(t)
    print(route_trace(form))

## Memoria e cache

La memoria non e' magia: e' stato vincolato. Quattro livelli distinti:
- **Stato di sessione** — dati correnti del ticket;
- **Memoria episodica** — cosa e' successo in run precedenti;
- **Prefix cache** — parti statiche del prompt che il provider serve a costo ridotto;
- **Response cache** — risposta intera salvata per query identiche (azzera la chiamata al provider).

In [ ]:
# [OFFLINE] Stato sessione + episodica + stima prefix cache
from llm_ops_v1.caching import estimate_support_triage_cache_demo
from llm_ops_v1.memory.episodic import EpisodicMemory
from llm_ops_v1.memory.short_term import InMemorySessionState

session = InMemorySessionState()
session.put(DEPS.ticket_id, "customer_tier", DEPS.customer_tier)
session.put(DEPS.ticket_id, "ticket_priority", DEPS.ticket_priority)

episodes = EpisodicMemory()
episodes.record(
    actor="support-agent", action="drafted_reply", summary="Bozza per ticket spedizione in ritardo."
)

est = estimate_support_triage_cache_demo(
    ticket_prompt=PROMPT,
    output=TRIAGE_OUTPUT,
    policy_snippets=DEPS.policy_snippets,
)

{
    "stato_sessione": session.snapshot(DEPS.ticket_id),
    "episodi": [e.summary for e in episodes.latest()],
    "cache_key": est.cache_key,
    "prefix_tokens": est.prefix_tokens,
    "input_tokens": est.input_tokens,
    "output_tokens": est.output_tokens,
    "uncached_cost_usd": est.uncached_cost.total_cost_usd,
    "cached_cost_usd": est.cached_cost.total_cost_usd,
    "savings_usd": est.delta.savings_usd,
    "savings_pct": est.delta.savings_pct,
}

In [ ]:
# [OFFLINE] Response cache — la seconda query non chiama il provider
from IPython.display import Markdown, display

from llm_ops_v1.caching.response_cache import InMemoryCache, ResponseCache, make_response_key

cache = ResponseCache(InMemoryCache())
key = make_response_key("system", list(DEPS.policy_snippets), PROMPT, "anthropic:claude-haiku-4-5")

print("Hit prima query :", await cache.get(key))
await cache.set(key, TRIAGE_OUTPUT)
hit = await cache.get(key)
print("Hit seconda query: cache hit ✓")
print()
display(Markdown(hit or ""))

## Eval — offline [OFFLINE]

In produzione non basta "sembra una buona risposta": serve una rubric ripetibile. Il `DeterministicJudge` e' zero-dependency, gira in CI senza API key, e serve come baseline deterministico prima del giudice LLM.

In [ ]:
# [OFFLINE] DeterministicJudge — valuta struttura output, non semantica
from IPython.display import Markdown, display

from llm_ops_v1.evals.deterministic import evaluate

cases = [
    ("Output agente (live)", PROMPT, TRIAGE_OUTPUT),
    ("TODO — output rotto", "billing question", "TODO: add logic here"),
    ("Troppo corto", "fattura doppio addebito", "ok"),
]
for label, prompt, output in cases:
    score = evaluate(prompt, output)
    flag = "✓ PASS" if score.passed else "✗ FAIL"
    print(f"{flag}  score={score.score}/10  [{label}]")
    display(Markdown(f"> {output[:200]}"))
    print()

## Eval — LLM-as-judge [LIVE]

Il `ClaudeJudge` usa un LLM per valutare la risposta secondo una rubric. E' non-deterministico e ha un costo per chiamata — si usa su campione in CI offline, non inline su ogni request in produzione.

In [ ]:
# [LIVE — richiede ANTHROPIC_API_KEY]
import os

from IPython.display import Markdown, display

from llm_ops_v1.evals.llm_judge import ClaudeJudge
from llm_ops_v1.evals.rubrics import SUPPORT_TRIAGE_RUBRIC

display(Markdown(f"**Rubric:** {SUPPORT_TRIAGE_RUBRIC}"))
display(Markdown("---"))
display(Markdown(f"**Output valutato:**\n\n{TRIAGE_OUTPUT}"))
display(Markdown("---"))

if os.getenv("ANTHROPIC_API_KEY"):
    score = await ClaudeJudge().judge_output(
        prompt=PROMPT, output=TRIAGE_OUTPUT, rubric=SUPPORT_TRIAGE_RUBRIC
    )
    display(
        Markdown(f"""
**Score:** {score.score}/10 — {"✓ PASS" if score.passed else "✗ FAIL"}

**Rationale:**

{score.rationale}
""")
    )
else:
    print("[offline] ANTHROPIC_API_KEY non impostata — salto.")

## Golden set + LAB [OFFLINE]

Il golden set e' il cuore del sistema di eval: 15 ticket etichettati con azione attesa e categoria. Il runner produce pass rate, action accuracy e score medio — questi sono i numeri che difendi in code review prima di un deploy.

In [ ]:
# [LIVE se ANTHROPIC_API_KEY — offline altrimenti]
# Con chiave: 15 chiamate reali all'agente, ~$0.02 totali su Haiku
import os

from IPython.display import Markdown, display

from llm_ops_v1.agents.base_agent import run_support_triage_agent
from llm_ops_v1.evals.datasets import load_golden
from llm_ops_v1.evals.deterministic import DeterministicJudge
from llm_ops_v1.evals.runner import run_eval

dataset = load_golden()
agent_fn = run_support_triage_agent if os.getenv("ANTHROPIC_API_KEY") else None
mode = "LIVE — agente reale" if agent_fn else "OFFLINE — output sintetico"
print(f"Modalita: {mode}  ({len(dataset)} esempi)")

summary = await run_eval(dataset, DeterministicJudge(), agent_fn=agent_fn)

rows = "\n".join(
    f"| {r.example_id} | {r.expected_action} | {r.detected_action} | "
    f"{'✓' if r.action_correct else '✗'} | {'✓ PASS' if r.passed else '✗ FAIL'} | {r.score} |"
    for r in summary.results
)
display(
    Markdown(f"""
**Modalita:** {mode}

| KPI | Valore |
|---|---|
| Pass rate | {summary.pass_rate:.0%} |
| Avg score | {summary.avg_score:.1f}/10 |
| Action accuracy | {summary.action_accuracy:.0%} |

| ID | Expected | Detected | Corretta | Giudice | Score |
|---|---|---|---|---|---|
{rows}
""")
)

## Drift e alerting [OFFLINE]

PSI (Population Stability Index) misura quanto e' cambiata la distribuzione degli input e delle azioni rispetto a una baseline. `AlertEngine` confronta i KPI contro soglie e emette WARN/CRITICAL — questi sono i segnali che finiscono nel tab Drift & Alert del dashboard.

In [ ]:
# [LIVE — 30 ticket in parallelo, risposta + LLM-as-judge in parallelo]
# Offline: DeterministicJudge. Live: ClaudeJudge dopo le risposte.
import asyncio, os, time
from datetime import UTC, datetime
from IPython.display import Markdown, display

from llm_ops_v1.agents.base_agent import run_triage_with_usage
from llm_ops_v1.evals.deterministic import evaluate as det_evaluate
from llm_ops_v1.evals.llm_judge import ClaudeJudge
from llm_ops_v1.evals.rubrics import SUPPORT_TRIAGE_RUBRIC
from llm_ops_v1.dashboard.models import DashboardRecord
from llm_ops_v1.evals.drift import compute_drift
from llm_ops_v1.monitoring.alerts import AlertEngine, AlertThresholds

TICKETS = [
    "Il mio ordine e in ritardo da 5 giorni.",
    "Ho ricevuto un doppio addebito in fattura.",
    "L app si blocca quando apro la sezione pagamenti.",
    "Voglio disdire il mio abbonamento annuale.",
    "Non riesco a fare il login dopo l aggiornamento.",
    "Enterprise: sistema SSO non funzionante per tutto il team.",
    "Dove trovo la ricevuta fiscale del mio acquisto?",
    "Il rimborso non e arrivato dopo 10 giorni.",
    "Posso aggiornare l indirizzo di spedizione?",
    "Il pacco risulta consegnato ma non l ho ricevuto.",
    "Ho bisogno di una fattura con i dati aziendali.",
    "Vorrei fare un reso ma il link non funziona.",
    "Account bloccato dopo troppi tentativi.",
    "Il codice promozionale non viene accettato.",
    "Sito non disponibile da stamattina.",
    "Come funziona il programma fedelta?",
    "Prodotto ricevuto diverso da quello ordinato.",
    "Non riesco a scaricare le istruzioni.",
    "Offerta applicata nel carrello ma non in fattura.",
    "Ho un problema con billing e anche account bloccato — urgente.",
    "Il mio ordine ha tracking fermo da 3 giorni.",
    "Rimborso promesso la settimana scorsa non arrivato.",
    "App crash ogni volta che apro notifiche.",
    "Abbonamento rinnovato automaticamente senza consenso.",
    "Team di 50 persone non riesce ad accedere al portale.",
    "Fattura mancante per l acquisto di ieri.",
    "Spedizione priority non arrivata entro i tempi SLA.",
    "Bug nella sezione report — dati errati.",
    "Voglio passare al piano enterprise.",
    "Help.",
]

live = bool(os.getenv("ANTHROPIC_API_KEY"))
judge = ClaudeJudge() if live else None
print(f"{'LIVE' if live else 'OFFLINE'} — {len(TICKETS)} ticket in parallelo...")
t_start = time.perf_counter()

# FASE 1: tutte le risposte in parallelo
async def triage_one(ticket, idx):
    t0 = time.perf_counter()
    r = await run_triage_with_usage(ticket)
    return idx, ticket, r.output, (time.perf_counter() - t0) * 1000, r.estimated

results = await asyncio.gather(*[triage_one(t, i) for i, t in enumerate(TICKETS)])
t_triage = time.perf_counter() - t_start
print(f"  Risposte: {len(results)} in {t_triage:.1f}s")

# FASE 2: tutti i giudizi in parallelo
async def judge_one(idx, ticket, output):
    if judge:
        score = await judge.judge_output(ticket, output, SUPPORT_TRIAGE_RUBRIC)
    else:
        score = det_evaluate(ticket, output)
    return idx, score

scores = await asyncio.gather(*[judge_one(idx, ticket, output) for idx, ticket, output, _, _ in results])
score_map = {idx: s for idx, s in scores}
t_total = time.perf_counter() - t_start
print(f"  Giudizi:  {len(scores)} in {t_total:.1f}s totali")

# Costruisci record
def detect_action(output):
    l = output.lower()
    if "escalate" in l: return "escalate"
    if "clarif" in l: return "ask_clarification"
    return "reply"

records = [
    DashboardRecord(
        run_id=f"parallel-{idx:02d}",
        prompt_preview=ticket,
        output_preview=output,
        latency_ms=latency,
        cost_usd=0.0,
        eval_score=float(score_map[idx].score),
        action=detect_action(output),
        estimated=estimated,
        timestamp=datetime.now(UTC),
    )
    for idx, ticket, output, latency, estimated in results
]

# Drift e alert
half = len(records) // 2
drift = compute_drift(records[:half], records[half:])
engine = AlertEngine(AlertThresholds(min_avg_score=5.0, max_p95_latency_ms=15000.0))
alerts = engine.evaluate(records, drift)

avg_score = sum(r.eval_score for r in records) / len(records)
avg_lat = sum(r.latency_ms for r in records) / len(records)
actions = {a: sum(1 for r in records if r.action == a) for a in ("reply","escalate","ask_clarification")}
pass_rate = sum(1 for idx, s in score_map.items() if s.passed) / len(score_map)

rows = "\n".join(
    f"| {idx:02d} | {ticket[:45]} | {detect_action(output)} | {score_map[idx].score} | {'✓' if score_map[idx].passed else '✗'} | {round(latency)}ms |"
    for idx, ticket, output, latency, _ in results
)
alert_rows = "\n".join(
    f"| **[{a.level}]** | `{a.metric}` | {a.value:.2f} | {a.threshold} |"
    for a in alerts
) if alerts else "✓ Nessun alert"

judge_label = "ClaudeJudge" if live else "DeterministicJudge"
display(Markdown(f"""
## Esecuzione parallela — {len(TICKETS)} ticket  ({judge_label})

| KPI | Valore |
|---|---|
| Tempo totale | {t_total:.1f}s |
| Avg latenza per ticket | {round(avg_lat)}ms |
| Avg score | {avg_score:.1f}/10 |
| Pass rate | {pass_rate:.0%} |
| reply | {actions["reply"]} |
| escalate | {actions["escalate"]} |
| ask_clarification | {actions["ask_clarification"]} |

| # | Ticket | Action | Score | Pass | Latenza |
|---|---|---|---|---|---|
{rows}

## Drift (prima metà → seconda metà)

| Metrica | Valore | Stato |
|---|---|---|
| PSI lunghezza prompt | {drift.psi_prompt_length.psi:.4f} | {"⚠" if drift.psi_prompt_length.drift else "✓"} |
| PSI azioni | {drift.psi_action.psi:.4f} | {"⚠" if drift.psi_action.drift else "✓"} |
| Score baseline | {drift.score_drift.baseline_avg:.1f} | — |
| Score corrente | {drift.score_drift.current_avg:.1f} | {"⚠" if drift.score_drift.drift else "✓"} |

## Alert ({len(alerts)})

{"| Livello | Metrica | Valore | Soglia |\n|---|---|---|---|\n" + alert_rows if alerts else alert_rows}
"""))


## Dashboard — dove entra Streamlit

Streamlit non e' la logica dell'agente: e' la superficie operativa che legge record costruiti dal codice.

Flusso: agente → `build_live_record` → `DashboardStore` → `build_summary` → Streamlit.  
Qui costruiamo un record dalla run appena fatta e vediamo i KPI che Streamlit mostrerebbe.

In [ ]:
# [OFFLINE]
from IPython.display import Markdown, display

from llm_ops_v1.dashboard.app import build_summary
from llm_ops_v1.dashboard.demo_runner import build_live_record
from llm_ops_v1.dashboard.store import DashboardStore

record = build_live_record(
    prompt=PROMPT,
    output=TRIAGE_OUTPUT,
    latency_ms=900,
    is_estimated=result.estimated,
)
store = DashboardStore([])
store.append(record)
summary = build_summary(store.snapshot())

kpi_rows = "\n".join(f"| {k} | {v} |" for k, v in summary.__dict__.items())
cost_label = "Costo (stima)" if record.estimated else "Costo reale"
display(
    Markdown(f"""
## KPI Streamlit

| Metrica | Valore |
|---|---|
{kpi_rows}

## Record — una run

| Campo | Valore |
|---|---|
| run_id | `{record.run_id}` |
| action | {record.action} |
| {cost_label} | ${record.cost_usd:.5f} |
| eval_score | {record.eval_score}/10 |
| latency_ms | {round(record.latency_ms)} ms |
| cache_hit | {record.cache_hit} |
| estimated | {record.estimated} |
""")
)

## Osservabilita — dove entra Langfuse [LIVE se chiavi presenti]

Langfuse e' il backend di tracing e osservabilita, non la dashboard Streamlit. Streamlit mostra una vista locale e didattica; Langfuse conserva trace, prompt, token, latenza, costi e run history.

Sintesi: notebook = walkthrough, Streamlit = cockpit operativo, Langfuse = scatola nera consultabile dopo e durante le run.

**Stack locale:** `make up` avvia Langfuse su `http://localhost:3001`.  
Configura l'utente locale tramite le variabili `LANGFUSE_INIT_USER_*` in `.env`.

In [ ]:
# [LIVE se LANGFUSE_PUBLIC_KEY e LANGFUSE_SECRET_KEY sono impostate — no-op altrimenti]
import os

from llm_ops_v1.evals.deterministic import evaluate
from llm_ops_v1.observability.langfuse_setup import flush_langfuse, push_eval_score

langfuse_on = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))
langfuse_label = "non configurato"
if langfuse_on:
    langfuse_label = "attivo su " + os.getenv("LANGFUSE_HOST", "")
print(f"Langfuse: {langfuse_label}")
print()
# La trace e' gia stata creata da @observe in run_triage_with_usage (cella scenario).
# Qui proviamo ad associare uno score deterministico alla trace corrente.
det = evaluate(PROMPT, TRIAGE_OUTPUT)
push_eval_score(
    name="triage_quality_deterministic", trace_id="", score=float(det.score), comment=det.rationale
)
flush_langfuse()

if langfuse_on:
    print(f"Score pushato: triage_quality_deterministic = {det.score}")
    print("Apri Langfuse > Traces per vedere la trace 'triage' creata dalla cella scenario.")

## Handoff runtime

Tre percorsi ripetibili dopo la sessione:

```bash
make stack          # tira su tutto: Docker + worker + dashboard
make notebook       # riapre questo notebook in JupyterLab
make demo           # stessa suite da terminale, senza UI
make load-test N=20 # 20 ticket in parallelo, multi-provider
uv run pytest -q    # regressioni su golden set e guardrails
```

Il notebook importa da `llm_ops_v1` — non copiare logica nelle celle. Se aggiungi un provider o una metrica, modifichi il package e riesegui le celle.

Navigazione: `docs/INDEX.md` per moduli e teoria. `docs/runbook.md` per incident durante la demo.